Okay so the plan for calibrating the thresholds for the HQ dataset.
1) just check the performance based on the optimal threshold based on validation data of standard dataset (this was now mistakenly calculated on test data!!! fixed now but needs rerun)
2) Simple threshold adjustment 
    1) Do predicts for validation set of HQ dataset (limited subset)
    2) Calculate optimal thresholds based on this
    3) Get test performance with these metrics
3) Fancy adjustment: Actually change the probabilities based on HQ data



In [1]:
import h5torch
import numpy as np
import torch
import pytorch_lightning as pl
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.calibration import calibration_curve
import warnings
from pytorch_lightning.callbacks import Callback
from TFBS_negatives.data import DataModule
import pytorch_lightning as pl
from TFBS_negatives.models import TFmodel
from pytorch_lightning.callbacks import ModelCheckpoint
import torch
import pytorch_lightning as pl
import warnings
import numpy as np
import os
from torchmetrics.classification import MultilabelAUROC, MultilabelAveragePrecision
from torchmetrics.functional.classification import multilabel_average_precision, binary_auroc, binary_average_precision, binary_accuracy, binary_matthews_corrcoef, binary_precision, binary_specificity, binary_recall
import gc
#

/data/home/natant/anaconda3/envs/Negs2/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

def get_predicts(model_ckpt_path, predict_set, device=3, idx=False):
    convert_dict = {
    "dinucl_sampled": "dinucl-sampled",
    "dinucl_shuffled": "dinucl-shuffled",
}
    filename = os.path.basename(model_ckpt_path)
    for old_name, new_name in convert_dict.items():
        filename = filename.replace(old_name, new_name)
    parts = filename.split('_')
    
    # Extract cell line (first part)
    cellline = parts[0]    
    cv_split = parts[-7].split('-')[1]
    neg_type = parts[-8]
    tf = '_'.join(parts[1:-8])
    Original_neg_mode = neg_type.replace('-', '_')

    if neg_type == "celltype":
        cellline_file = f"/data/home/natant/Negatives/Data/Encode690/ENCODE_hg38_subset_101bp_celltypes_ATAC_H5_all_chr copy/{cellline}.h5t"
    else:
        cellline_file = f"/data/home/natant/Negatives/Data/Encode690/ENCODE_hg38_subset_101bp_celltypes_ATAC_H5_all_chr/{cellline}.h5t"


    Dmod = DataModule(cellline_file, TF=tf, batch_size=256, neg_mode=Original_neg_mode, cross_val_set=int(cv_split))
    Dmod.setup(predict_set=predict_set)
    model = TFmodel.load_from_checkpoint(model_ckpt_path)

    trainer = pl.Trainer(
        max_steps=5_000_000,
        accelerator="gpu",
        devices=[device]
    )
    if idx is False:
        predict_outputs = trainer.predict(model, Dmod)
    else:
        predict_outputs = trainer.predict(model, Dmod)[idx]
    return predict_outputs

In [3]:
# Define the cell line file path
cellline_file = "/data/home/natant/Negatives/Data/Encode690/ENCODE_hg38_subset_101bp_celltypes_ATAC_H5_all_chr/GM12878.h5t"

# Define the folder path
folder_path = "/data/home/natant/Negatives/Runs/Review_rerun"

# Get all files ending in .ckpt
ckpt_files = [f for f in os.listdir(folder_path) if f.endswith('.ckpt')]

# convert the names (OLD NAMING SCHEME)
convert_dict = {
    "dinucl_sampled": "dinucl-sampled",
    "dinucl_shuffled": "dinucl-shuffled",
}

# Create output folder if it doesn't exist
output_folder = "/data/home/natant/Negatives/Runs/Review_rerun/Calibration_plots/"
os.makedirs(output_folder, exist_ok=True)

# Define variables
cellline = 'GM12878'
neg_type_order = ['dinucl-sampled', 'neighbors', 'celltype', 'shuffled', 'dinucl-shuffled']




In [4]:
# Parse each filename to extract information (OLD NAMING SCHEME AGAIN)
data = []
negative_types = ["dinucl-sampled", "celltype", "shuffled", "dinucl-shuffled", "neighbors"]
for filename in ckpt_files:   
    old_filename = filename
    for old_name, new_name in convert_dict.items():
        filename = filename.replace(old_name, new_name)
    parts = filename.split('_')
    neg_type = parts[-8]
    if neg_type not in negative_types:
        neg_type = "HQ"
        tf = '_'.join(parts[1:-7])
    else:
        tf = '_'.join(parts[1:-8])
        neg_type = parts[-8]

    
    # Extract cell line (first part)
    cellline = parts[0] 
    cv_split = parts[-7].split('-')[1]


    
    
    data.append({
        'filename': filename,
        'file_path': os.path.join(folder_path, old_filename),
        'cellline': cellline,
        'TF': tf,
        'negative_type': neg_type,
        'cv_split': cv_split
    })

# Create DataFrame
df_ckpt = pd.DataFrame(data)

In [5]:
df_ckpt

,filename,file_path,cellline,TF,negative_type,cv_split
0,GM12878_TCF12_dinucl-sampled_CV-4_20251031_02:...,/data/home/natant/Negatives/Runs/Review_rerun/...,GM12878,TCF12,dinucl-sampled,4
1,GM12878_NFIC_(SC-81335)_celltype_CV-4_20251103...,/data/home/natant/Negatives/Runs/Review_rerun/...,GM12878,NFIC_(SC-81335),celltype,4
2,GM12878_ETS1_celltype_CV-0_20251103_21:31_epoc...,/data/home/natant/Negatives/Runs/Review_rerun/...,GM12878,ETS1,celltype,0
3,GM12878_SIX5_shuffled_CV-2_20251030_19:51_epoc...,/data/home/natant/Negatives/Runs/Review_rerun/...,GM12878,SIX5,shuffled,2
4,GM12878_ZBTB33_celltype_CV-0_20251103_21:12_ep...,/data/home/natant/Negatives/Runs/Review_rerun/...,GM12878,ZBTB33,celltype,0
...,...,...,...,...,...,...
1423,GM12878_TBP_shuffled_CV-4_20251030_16:28_epoch...,/data/home/natant/Negatives/Runs/Review_rerun/...,GM12878,TBP,shuffled,4
1424,GM12878_NFIC_(SC-81335)_celltype_CV-1_20251103...,/data/home/natant/Negatives/Runs/Review_rerun/...,GM12878,NFIC_(SC-81335),celltype,1
1425,HepG2_FOXA1_(SC-101058)_neighbors_CV-5_2025103...,/data/home/natant/Negatives/Runs/Review_rerun/...,HepG2,FOXA1_(SC-101058),neighbors,5
1426,GM12878_ATF2_(SC-81188)_shuffled_CV-3_20251030...,/data/home/natant/Negatives/Runs/Review_rerun/...,GM12878,ATF2_(SC-81188),shuffled,3


In [14]:
counts = df_ckpt.groupby(['TF', 'negative_type'])["file_path"].nunique()

In [18]:
from tqdm import tqdm
num_negs = len(neg_type_order)
TF_list = df_ckpt['TF'].unique().tolist()
all_results = []

In [20]:
for TF in tqdm(TF_list, desc="Processing TFs", position=0, leave=True):
    for cross_val in tqdm(range(5), desc="Processing CV splits", position=1, leave=True):
        selected_runs = df_ckpt[
        (df_ckpt['TF'] == TF) & 
        (df_ckpt['cellline'] == cellline) & 
        (df_ckpt['cv_split'] == str(cross_val)) & 
        (df_ckpt["negative_type"] != "HQ")
        ]
        for neg_type in tqdm(neg_type_order, desc="Processing negative types", position=2, leave=True):
            if selected_runs[selected_runs["negative_type"] == neg_type].empty:
                print(f"\033[91mSkipping TF:{TF}, CV:{cross_val}, NEG:{neg_type} as no run found.\033[0m")
                continue
            model_ckpt_path = selected_runs[selected_runs["negative_type"] == neg_type]["file_path"].values[0]
            pass

Processing negative types: 100%|██████████| 5/5 [00:00<00:00, 670.47it/s]

Processing negative types: 100%|██████████| 5/5 [00:00<00:00, 954.77it/s]

Processing negative types: 100%|██████████| 5/5 [00:00<00:00, 1120.93it/s]

Processing negative types: 100%|██████████| 5/5 [00:00<00:00, 1204.91it/s]

Processing CV splits: 100%|██████████| 5/5 [00:00<00:00, 67.14it/s]

Processing negative types: 100%|██████████| 5/5 [00:00<00:00, 821.93it/s]

Processing negative types: 100%|██████████| 5/5 [00:00<00:00, 1243.35it/s]

Processing negative types: 100%|██████████| 5/5 [00:00<00:00, 1165.93it/s]

Processing negative types: 100%|██████████| 5/5 [00:00<00:00, 1091.47it/s]

Processing negative types: 100%|██████████| 5/5 [00:00<00:00, 934.14it/s]

Processing negative types: 100%|██████████| 5/5 [00:00<00:00, 1135.31it/s]

Processing negative types: 100%|██████████| 5/5 [00:00<00:00, 1079.95it/s]

Processing negative types: 100%|██████████| 5/5 [00:00<00:00, 1251.96it/s]

Processing CV splits: 1

Skipping TF:MafK_(ab50322), CV:0, NEG:dinucl-sampled as no run found.
Skipping TF:MafK_(ab50322), CV:0, NEG:neighbors as no run found.
Skipping TF:MafK_(ab50322), CV:0, NEG:celltype as no run found.
Skipping TF:MafK_(ab50322), CV:0, NEG:shuffled as no run found.
Skipping TF:MafK_(ab50322), CV:0, NEG:dinucl-shuffled as no run found.



Processing negative types: 100%|██████████| 5/5 [00:00<00:00, 1850.48it/s]


Skipping TF:MafK_(ab50322), CV:1, NEG:dinucl-sampled as no run found.
Skipping TF:MafK_(ab50322), CV:1, NEG:neighbors as no run found.
Skipping TF:MafK_(ab50322), CV:1, NEG:celltype as no run found.
Skipping TF:MafK_(ab50322), CV:1, NEG:shuffled as no run found.
Skipping TF:MafK_(ab50322), CV:1, NEG:dinucl-shuffled as no run found.



Processing negative types: 100%|██████████| 5/5 [00:00<00:00, 1857.86it/s]


...


Processing negative types: 100%|██████████| 5/5 [00:00<00:00, 1250.46it/s]


Skipping TF:FOXA1_(SC-101058), CV:3, NEG:dinucl-sampled as no run found.
Skipping TF:FOXA1_(SC-101058), CV:3, NEG:neighbors as no run found.
Skipping TF:FOXA1_(SC-101058), CV:3, NEG:celltype as no run found.
Skipping TF:FOXA1_(SC-101058), CV:3, NEG:shuffled as no run found.
Skipping TF:FOXA1_(SC-101058), CV:3, NEG:dinucl-shuffled as no run found.



Processing CV splits: 100%|██████████| 5/5 [00:00<00:00, 64.29it/s]


Skipping TF:FOXA1_(SC-101058), CV:4, NEG:dinucl-sampled as no run found.
Skipping TF:FOXA1_(SC-101058), CV:4, NEG:neighbors as no run found.
Skipping TF:FOXA1_(SC-101058), CV:4, NEG:celltype as no run found.
Skipping TF:FOXA1_(SC-101058), CV:4, NEG:shuffled as no run found.
Skipping TF:FOXA1_(SC-101058), CV:4, NEG:dinucl-shuffled as no run found.



Processing negative types: 100%|██████████| 5/5 [00:00<00:00, 609.67it/s]

Processing negative types: 100%|██████████| 5/5 [00:00<00:00, 907.78it/s]

Processing negative types: 100%|██████████| 5/5 [00:00<00:00, 974.24it/s]

Processing negative types: 100%|██████████| 5/5 [00:00<00:00, 1086.21it/s]

Processing TFs: 100%|██████████| 38/38 [00:02<00:00, 13.06it/s]


In [16]:
counts

TF                   negative_type  
ATF2_(SC-81188)      HQ                 6
                     celltype           6
                     dinucl-sampled     6
                     dinucl-shuffled    6
                     neighbors          6
                                       ..
Znf143_(16618-1-AP)  celltype           6
                     dinucl-sampled     6
                     dinucl-shuffled    6
                     neighbors          6
                     shuffled           6
Name: file_path, Length: 222, dtype: int64

In [ ]:
num_negs = len(neg_type_order)
TF_list = df_ckpt['TF'].unique().tolist()
all_results = []

for TF in TF_list:
    for cross_val in range(5):
        selected_runs = df_ckpt[
        (df_ckpt['TF'] == TF) & 
        (df_ckpt['cellline'] == cellline) & 
        (df_ckpt['cv_split'] == str(cross_val)) & 
        (df_ckpt["negative_type"] != "HQ")
        ]
        for neg_type in neg_type_order:
            model_ckpt_path = selected_runs[selected_runs["negative_type"] == neg_type]["file_path"].values[0]
            # validation run
            batch_list = get_predicts(model_ckpt_path, device=3, predict_set="val")
            std_dataset = batch_list[0]
            HQ_dataset = batch_list[1]
            # std dataset
            y_hat_all = [] 
            y_all = []
            for item in std_dataset:
                if item is None:
                    continue
                logits = item["logits"].detach().cpu().numpy().reshape(-1)
                targ = item["target"].detach().cpu().numpy().reshape(-1)
                y_hat_all.append(logits)
                y_all.append(targ)
            y_hat_all = np.concatenate(y_hat_all, axis=0)
            y_all = np.concatenate(y_all, axis=0) 

            # threshold metrics
            thresholds = torch.arange(0.01, 1, 0.01, dtype=torch.float64)
            best_score = {"MCC": -1, "Precision": -1, "Specificity": -1, "Accuracy": -1, "Recall": -1} 
            best_thresholds = {"MCC": 0.5, "Precision": 0.5, "Specificity": 0.5, "Accuracy": 0.5, "Recall": 0.5}
            for t in thresholds:
                t=float(t)
                mcc = binary_matthews_corrcoef(y_hat_all, y_all, threshold=t)
                prec = binary_precision(y_hat_all, y_all, threshold=t)
                spec = binary_specificity(y_hat_all, y_all, threshold=t)
                acc = binary_accuracy(y_hat_all, y_all, threshold=t)
                rec = binary_recall(y_hat_all, y_all, threshold=t)

                if mcc > best_score["MCC"]:
                    best_score["MCC"], best_thresholds["MCC"] = mcc, t
                if prec > best_score["Precision"]:
                    best_score["Precision"], best_thresholds["Precision"] = prec, t
                if spec > best_score["Specificity"]:
                    best_score["Specificity"], best_thresholds["Specificity"] = spec, t
                if acc > best_score["Accuracy"]:
                    best_score["Accuracy"], best_thresholds["Accuracy"] = acc, t
                if rec > best_score["Recall"]:
                    best_score["Recall"], best_thresholds["Recall"] = rec, t

            # HQ dataset
            y_hat_all = [] 
            y_all = []
            for item in HQ_dataset:
                if item is None:
                    continue
                logits = item["logits"].detach().cpu().numpy().reshape(-1)
                targ = item["target"].detach().cpu().numpy().reshape(-1)
                y_hat_all.append(logits)
                y_all.append(targ)
            y_hat_all = np.concatenate(y_hat_all, axis=0)
            y_all = np.concatenate(y_all, axis=0)

            AUROC = binary_auroc(y_hat_all, y_all)
            Average_precision = binary_average_precision(y_hat_all, y_all)

            # threshold metrics
            thresholds = torch.arange(0.01, 1, 0.01, dtype=torch.float64)
            best_score_HQ = {"MCC": -1, "Precision": -1, "Specificity": -1, "Accuracy": -1, "Recall": -1} 
            best_thresholds_HQ = {"MCC": 0.5, "Precision": 0.5, "Specificity": 0.5, "Accuracy": 0.5, "Recall": 0.5}
            for t in thresholds:
                t=float(t)
                mcc = binary_matthews_corrcoef(y_hat_all, y_all, threshold=t)
                prec = binary_precision(y_hat_all, y_all, threshold=t)
                spec = binary_specificity(y_hat_all, y_all, threshold=t)
                acc = binary_accuracy(y_hat_all, y_all, threshold=t)
                rec = binary_recall(y_hat_all, y_all, threshold=t)

                if mcc > best_score_HQ["MCC"]:
                    best_score_HQ["MCC"], best_thresholds_HQ["MCC"] = mcc, t
                if prec > best_score_HQ["Precision"]:
                    best_score_HQ["Precision"], best_thresholds_HQ["Precision"] = prec, t
                if spec > best_score_HQ["Specificity"]:
                    best_score_HQ["Specificity"], best_thresholds_HQ["Specificity"] = spec, t
                if acc > best_score_HQ["Accuracy"]:
                    best_score_HQ["Accuracy"], best_thresholds_HQ["Accuracy"] = acc, t
                if rec > best_score_HQ["Recall"]:
                    best_score_HQ["Recall"], best_thresholds_HQ["Recall"] = rec, t

            # test run
            batch_list = get_predicts(model_ckpt_path, device=3, predict_set="test")
            std_dataset = batch_list[0]
            HQ_dataset = batch_list[1]
            # std dataset
            y_hat_all = [] 
            y_all = []
            for item in std_dataset:
                if item is None:
                    continue
                logits = item["logits"].detach().cpu().numpy().reshape(-1)
                targ = item["target"].detach().cpu().numpy().reshape(-1)
                y_hat_all.append(logits)
                y_all.append(targ)
            y_hat_all = np.concatenate(y_hat_all, axis=0)
            y_all = np.concatenate(y_all, axis=0)

            test_scores = {}
            test_scores["AUROC"] = binary_auroc(y_hat_all, y_all)
            test_scores["AvP"] = binary_average_precision(y_hat_all, y_all)
            test_scores["MCC"] = binary_matthews_corrcoef(y_hat_all, y_all, threshold=best_thresholds["MCC"])
            test_scores["Precision"] = binary_precision(y_hat_all, y_all, threshold=best_thresholds["Precision"])
            test_scores["Specificity"] = binary_specificity(y_hat_all, y_all, threshold=best_thresholds["Specificity"])
            test_scores["Accuracy"] = binary_accuracy(y_hat_all, y_all, threshold=best_thresholds["Accuracy"])
            test_scores["Recall"] = binary_recall(y_hat_all, y_all, threshold=best_thresholds["Recall"])
            # HQ dataset
            y_hat_all = [] 
            y_all = []
            for item in std_dataset:
                if item is None:
                    continue
                logits = item["logits"].detach().cpu().numpy().reshape(-1)
                targ = item["target"].detach().cpu().numpy().reshape(-1)
                y_hat_all.append(logits)
                y_all.append(targ)
            y_hat_all = np.concatenate(y_hat_all, axis=0)
            y_all = np.concatenate(y_all, axis=0)
            test_scores_HQ = {}
            test_scores_HQ["AUROC"] = binary_auroc(y_hat_all, y_all)
            test_scores_HQ["AvP"] = binary_average_precision(y_hat_all, y_all)
            test_scores_HQ["MCC"] = binary_matthews_corrcoef(y_hat_all, y_all, threshold=best_thresholds["MCC"])
            test_scores_HQ["Precision"] = binary_precision(y_hat_all, y_all, threshold=best_thresholds["Precision"])
            test_scores_HQ["Specificity"] = binary_specificity(y_hat_all, y_all, threshold=best_thresholds["Specificity"])
            test_scores_HQ["Accuracy"] = binary_accuracy(y_hat_all, y_all, threshold=best_thresholds["Accuracy"])
            test_scores_HQ["Recall"] = binary_recall(y_hat_all, y_all, threshold=best_thresholds["Recall"])

            test_scores_HQ_adjusted = {}
            test_scores_HQ_adjusted["AUROC"] = binary_auroc(y_hat_all, y_all)
            test_scores_HQ_adjusted["AvP"] = binary_average_precision(y_hat_all, y_all)
            test_scores_HQ_adjusted["MCC"] = binary_matthews_corrcoef(y_hat_all, y_all, threshold=best_thresholds_HQ["MCC"])
            test_scores_HQ_adjusted["Precision"] = binary_precision(y_hat_all, y_all, threshold=best_thresholds_HQ["Precision"])
            test_scores_HQ_adjusted["Specificity"] = binary_specificity(y_hat_all, y_all, threshold=best_thresholds_HQ["Specificity"])
            test_scores_HQ_adjusted["Accuracy"] = binary_accuracy(y_hat_all, y_all, threshold=best_thresholds_HQ["Accuracy"])
            test_scores_HQ_adjusted["Recall"] = binary_recall(y_hat_all, y_all, threshold=best_thresholds_HQ["Recall"])


            results_dict = {
                "TF": TF,
                "cross_val": cross_val,
                "neg_type": neg_type,
                "cell_line": cellline}

            results_dict.update({
                f"val_best_thresholds_{k}": v for k, v in best_thresholds.items()
            })
            results_dict.update({
                f"val_best_scores_{k}": v for k, v in best_score.items()
            })
            results_dict.update({
                f"val_best_thresholds_HQ_{k}": v for k, v in best_thresholds_HQ.items()
            })
            results_dict.update({
                f"val_best_scores_HQ_{k}": v for k, v in best_score_HQ.items()
            })
            results_dict.update({
                f"test_scores_{k}": v for k, v in test_scores.items()
            })
            results_dict.update({
                f"test_scores_HQ_{k}": v for k, v in test_scores_HQ.items()
            })
            results_dict.update({
                f"test_scores_HQ_adjusted_{k}": v for k, v in test_scores_HQ_adjusted.items()
            })

            all_results.append(results_dict)

In [ ]:
results_df = pd.DataFrame(all_results)
results_df.to_pickle(os.path.join(output_folder, "calibration_results.pkl"))

In [ ]:
model_ckpt_path = selected_runs[selected_runs["negative_type"] == neg_type]["file_path"].values[0]
batch_list = get_predicts(model_ckpt_path, device=3, predict_set="val")
std_dataset = batch_list[0]
HQ_dataset = batch_list[1]

In [ ]:
# std dataset
y_hat_all = [] 
y_all = []
for item in std_dataset:
    if item is None:
        continue
    logits = item["logits"].detach().cpu().numpy().reshape(-1)
    targ = item["target"].detach().cpu().numpy().reshape(-1)
    y_hat_all.append(logits)
    y_all.append(targ)
y_hat_all = np.concatenate(y_hat_all, axis=0)
y_all = np.concatenate(y_all, axis=0) 

# threshold metrics
thresholds = torch.arange(0.01, 1, 0.01, dtype=torch.float64)
best_score = {"MCC": -1, "Precision": -1, "Specificity": -1, "Accuracy": -1, "Recall": -1} 
best_thresholds = {"MCC": 0.5, "Precision": 0.5, "Specificity": 0.5, "Accuracy": 0.5, "Recall": 0.5}
for t in thresholds:
    t=float(t)
    mcc = binary_matthews_corrcoef(y_hat_all, y_all, threshold=t)
    prec = binary_precision(y_hat_all, y_all, threshold=t)
    spec = binary_specificity(y_hat_all, y_all, threshold=t)
    acc = binary_accuracy(y_hat_all, y_all, threshold=t)
    rec = binary_recall(y_hat_all, y_all, threshold=t)

    if mcc > best_score["MCC"]:
        best_score["MCC"], best_thresholds["MCC"] = mcc, t
    if prec > best_score["Precision"]:
        best_score["Precision"], best_thresholds["Precision"] = prec, t
    if spec > best_score["Specificity"]:
        best_score["Specificity"], best_thresholds["Specificity"] = spec, t
    if acc > best_score["Accuracy"]:
        best_score["Accuracy"], best_thresholds["Accuracy"] = acc, t
    if rec > best_score["Recall"]:
        best_score["Recall"], best_thresholds["Recall"] = rec, t



In [ ]:
# HQ dataset
y_hat_all = [] 
y_all = []
for item in HQ_dataset:
    if item is None:
        continue
    logits = item["logits"].detach().cpu().numpy().reshape(-1)
    targ = item["target"].detach().cpu().numpy().reshape(-1)
    y_hat_all.append(logits)
    y_all.append(targ)
y_hat_all = np.concatenate(y_hat_all, axis=0)
y_all = np.concatenate(y_all, axis=0)

AUROC = binary_auroc(y_hat_all, y_all)
Average_precision = binary_average_precision(y_hat_all, y_all)

# threshold metrics
thresholds = torch.arange(0.01, 1, 0.01, dtype=torch.float64)
best_score_HQ = {"MCC": -1, "Precision": -1, "Specificity": -1, "Accuracy": -1, "Recall": -1} 
best_thresholds_HQ = {"MCC": 0.5, "Precision": 0.5, "Specificity": 0.5, "Accuracy": 0.5, "Recall": 0.5}
for t in thresholds:
    t=float(t)
    mcc = binary_matthews_corrcoef(y_hat_all, y_all, threshold=t)
    prec = binary_precision(y_hat_all, y_all, threshold=t)
    spec = binary_specificity(y_hat_all, y_all, threshold=t)
    acc = binary_accuracy(y_hat_all, y_all, threshold=t)
    rec = binary_recall(y_hat_all, y_all, threshold=t)

    if mcc > best_score_HQ["MCC"]:
        best_score_HQ["MCC"], best_thresholds_HQ["MCC"] = mcc, t
    if prec > best_score_HQ["Precision"]:
        best_score_HQ["Precision"], best_thresholds_HQ["Precision"] = prec, t
    if spec > best_score_HQ["Specificity"]:
        best_score_HQ["Specificity"], best_thresholds_HQ["Specificity"] = spec, t
    if acc > best_score_HQ["Accuracy"]:
        best_score_HQ["Accuracy"], best_thresholds_HQ["Accuracy"] = acc, t
    if rec > best_score_HQ["Recall"]:
        best_score_HQ["Recall"], best_thresholds_HQ["Recall"] = rec, t

In [ ]:
batch_list = get_predicts(model_ckpt_path, device=3, predict_set="test")
std_dataset = batch_list[0]
HQ_dataset = batch_list[1]
# std dataset
y_hat_all = [] 
y_all = []
for item in std_dataset:
    if item is None:
        continue
    logits = item["logits"].detach().cpu().numpy().reshape(-1)
    targ = item["target"].detach().cpu().numpy().reshape(-1)
    y_hat_all.append(logits)
    y_all.append(targ)
y_hat_all = np.concatenate(y_hat_all, axis=0)
y_all = np.concatenate(y_all, axis=0)

test_scores = {}
test_scores["AUROC"] = binary_auroc(y_hat_all, y_all)
test_scores["AvP"] = binary_average_precision(y_hat_all, y_all)
test_scores["MCC"] = binary_matthews_corrcoef(y_hat_all, y_all, threshold=best_thresholds["MCC"])
test_scores["Precision"] = binary_precision(y_hat_all, y_all, threshold=best_thresholds["Precision"])
test_scores["Specificity"] = binary_specificity(y_hat_all, y_all, threshold=best_thresholds["Specificity"])
test_scores["Accuracy"] = binary_accuracy(y_hat_all, y_all, threshold=best_thresholds["Accuracy"])
test_scores["Recall"] = binary_recall(y_hat_all, y_all, threshold=best_thresholds["Recall"])
# HQ dataset
y_hat_all = [] 
y_all = []
for item in std_dataset:
    if item is None:
        continue
    logits = item["logits"].detach().cpu().numpy().reshape(-1)
    targ = item["target"].detach().cpu().numpy().reshape(-1)
    y_hat_all.append(logits)
    y_all.append(targ)
y_hat_all = np.concatenate(y_hat_all, axis=0)
y_all = np.concatenate(y_all, axis=0)
test_scores_HQ = {}
test_scores_HQ["AUROC"] = binary_auroc(y_hat_all, y_all)
test_scores_HQ["AvP"] = binary_average_precision(y_hat_all, y_all)
test_scores_HQ["MCC"] = binary_matthews_corrcoef(y_hat_all, y_all, threshold=best_thresholds["MCC"])
test_scores_HQ["Precision"] = binary_precision(y_hat_all, y_all, threshold=best_thresholds["Precision"])
test_scores_HQ["Specificity"] = binary_specificity(y_hat_all, y_all, threshold=best_thresholds["Specificity"])
test_scores_HQ["Accuracy"] = binary_accuracy(y_hat_all, y_all, threshold=best_thresholds["Accuracy"])
test_scores_HQ["Recall"] = binary_recall(y_hat_all, y_all, threshold=best_thresholds["Recall"])

test_scores_HQ_adjusted = {}
test_scores_HQ_adjusted["AUROC"] = binary_auroc(y_hat_all, y_all)
test_scores_HQ_adjusted["AvP"] = binary_average_precision(y_hat_all, y_all)
test_scores_HQ_adjusted["MCC"] = binary_matthews_corrcoef(y_hat_all, y_all, threshold=best_thresholds_HQ["MCC"])
test_scores_HQ_adjusted["Precision"] = binary_precision(y_hat_all, y_all, threshold=best_thresholds_HQ["Precision"])
test_scores_HQ_adjusted["Specificity"] = binary_specificity(y_hat_all, y_all, threshold=best_thresholds_HQ["Specificity"])
test_scores_HQ_adjusted["Accuracy"] = binary_accuracy(y_hat_all, y_all, threshold=best_thresholds_HQ["Accuracy"])
test_scores_HQ_adjusted["Recall"] = binary_recall(y_hat_all, y_all, threshold=best_thresholds_HQ["Recall"])


In [ ]:
results_dict = {
    "TF": TF,
    "cross_val": cross_val,
    "neg_type": neg_type,
    "cell_line": cellline}

results_dict.update({
    f"val_best_thresholds_{k}": v for k, v in best_thresholds.items()
})
results_dict.update({
    f"val_best_scores_{k}": v for k, v in best_score.items()
})
results_dict.update({
    f"val_best_thresholds_HQ_{k}": v for k, v in best_thresholds_HQ.items()
})
results_dict.update({
    f"val_best_scores_HQ_{k}": v for k, v in best_score_HQ.items()
})
results_dict.update({
    f"test_scores_{k}": v for k, v in test_scores.items()
})
results_dict.update({
    f"test_scores_HQ_{k}": v for k, v in test_scores_HQ.items()
})
results_dict.update({
    f"test_scores_HQ_adjusted_{k}": v for k, v in test_scores_HQ_adjusted.items()
})

all_results.append(results_dict)